In [9]:
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, DirectoryLoader  #load directories
from langchain_text_splitters import RecursiveCharacterTextSplitter  #text splitter
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_openai import OpenAIEmbeddings  #Embedding model
from langchain_chroma import Chroma  #Vector DB
from dotenv import load_dotenv
import re

In [ ]:
load_dotenv()
url = "https://www.youtube.com/watch?v=bCz4OMemCcA"

In [11]:
def fetch_video_id(url:str):
    video_id = url.split("v=")[-1]
    print(f"--- Video ID: {video_id} ---")
    return video_id

In [12]:
def sanitize_text(text):
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r' +', ' ', text)
    text = text.strip()
    return text

In [13]:
def sanitize_build_documents(video_id:str):
    ytt = YouTubeTranscriptApi()
    segments = ytt.fetch(video_id)

    print(segments)
    # Sanitize + build Documents
    docs = []
    text = ""
    start_time = segments[0].start
    for seg in segments:
        text += " " + sanitize_text(seg.text)

        if not text or text.startswith("["):
            continue

    print(text)
    docs.append(Document(
            page_content=text,
            metadata={"video_id": video_id, "start": start_time, "source": url}
        ))
    print(f"{len(docs)} documents created")
    print(docs)
    return docs

In [14]:
def ingestion(url:str):
    video_id = fetch_video_id(url)
    docs = sanitize_build_documents(video_id)
    return docs

In [15]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)

video_id = fetch_video_id(url)
docs = ingestion(url)
chunks = splitter.split_documents(docs)
print(f"\n--- Len chunks {len(chunks)} ---")
print(f"\n--- chunk {chunks[1]}")

# vectorstore = Chroma(
#         persist_directory="db/chroma_db",
#         embedding_function=embeddings,
#         collection_metadata={"hnsw:space": "cosine"}
#     )

vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="db/chroma_db",
        collection_metadata={"hnsw:space": "cosine"}
    )

existing = vectorstore.get(where={"video_id": {"$eq": video_id}})

# if existing["ids"]:
#     print(f"Already ingested {video_id}, skipping.")
# else:
#     vectorstore.add_documents(chunks)


--- Video ID: bCz4OMemCcA ---
--- Video ID: bCz4OMemCcA ---
FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hello guys welcome to my video about the', start=0.0, duration=4.68), FetchedTranscriptSnippet(text='Transformer and this is actually the', start=2.22, duration=5.4), FetchedTranscriptSnippet(text='person 2.0 of my series on the', start=4.68, duration=5.94), FetchedTranscriptSnippet(text='Transformer I had a previous video in', start=7.62, duration=4.8), FetchedTranscriptSnippet(text='which I talked about the Transformer but', start=10.62, duration=3.66), FetchedTranscriptSnippet(text='the audio quality was not good and as', start=12.42, duration=4.32), FetchedTranscriptSnippet(text='suggested by my viewers as the video was', start=14.28, duration=5.579), FetchedTranscriptSnippet(text='really uh had a huge success the viewers', start=16.74, duration=4.98), FetchedTranscriptSnippet(text='suggested me to to improve their audio', start=19.859, duration=3.661), FetchedTran

In [16]:
print(vectorstore._collection.count())

9120
